## Purpose
This notebook **adds drug metadata to an existing NDC list**.

Given an input Excel file with an **`NDC`** column, it looks up each NDC against public APIs and produces **chunked CSV outputs** suitable for later merging back to your master table.

The lookup order is designed to **maximize match rate** while being **gentle to public endpoints**:
1. openFDA `drug/label` (archived labels)
2. RxNav `ndcstatus` (fallback)

## Inputs
- **`INPUT_XLSX`**: Excel file path
- **`NDC_COLUMN`**: `"NDC"` 

## Output
- A folder **`OUT_DIR`** containing one CSV per chunk:
  - `ndc_metadata_chunk_001.csv`, `ndc_metadata_chunk_002.csv`, ...

Each output row contains:
- `NDC_11`: 11-digit, digits-only NDC (string). If Excel dropped leading zeros, we left-pad.
- `lookup_source`: one of `FDA_LABEL_ARCHIVE`, `RXNORM`, `NOT_FOUND`
- `generic_name`, `brand_name`, `labeler_name` (when available)

## How to (re)run safely
- The notebook is **resume-safe by chunk**.
- If a chunk output file already exists, that chunk is skipped.
  - To re-run from scratch, delete the `OUT_DIR` folder.
  - To re-run only a specific chunk, delete just that chunk CSV.


In [ ]:
import os
import re
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry

# ---------------------------------------------------------
# NDC -> drug metadata
# FDA label archive first, RxNorm fallback
# ---------------------------------------------------------
# Workflow:
#   1. Read NDCs from Excel
#   2. Normalize to 11-digit strings
#   3. Query openFDA drug labels
#   4. Fallback to RxNorm if no FDA hit
#   5. Save chunked CSV outputs
#
# Why this simplified approach?
# - Faster than FDA package/product lookup
# - FDA label archive still captures many active drugs
# - RxNorm provides strong fallback coverage
# - Good for generic_name extraction workflows

# -----------------------------
# User-configurable parameters
# -----------------------------
# Input Excel file that contains an NDC column.
INPUT_XLSX = "Asthma and AR NDCs_4.29.2025.xlsx"

# Column name is known/fixed for this workflow.
NDC_COLUMN = "NDC"

# Output directory for chunk CSVs (resume-safe).
OUT_DIR = "ndc_drug_info_chunks"
os.makedirs(OUT_DIR, exist_ok=True)

# Chunk size controls API pacing and restart granularity.
CHUNK_SIZE = 100

# ---------------------------------------------------------
# HTTP session with retry handling
# ---------------------------------------------------------
def get_session():

    session = requests.Session()

    retries = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504]
    )

    session.mount(
        "https://",
        HTTPAdapter(max_retries=retries)
    )

    return session

http = get_session()

# ---------------------------------------------------------
# Normalize NDC values to 11-digit digits-only strings
# ---------------------------------------------------------
# Why we normalize:
# - Excel often imports NDC codes as numbers, which can drop leading zeros.
# - Downstream FDA/RxNorm calls expect stable string identifiers.
#
# What this function does:
# - Converts the input cell to a string
# - Removes all non-digits
# - Left-pads to 11 digits if the value is shorter (common when leading zeros were dropped)
# - Returns None for empty/invalid cells
#
# NOTE: This is intentionally permissive (1..11 digits) because upstream data quality varies.
# If you want stricter validation, tighten this to exactly 10/11 digits with known rules.
def clean_ndc11(value):

    # Handle missing cells (NaN/None)
    if value is None:
        return None

    # Convert to string, trim whitespace
    s = str(value).strip()

    # Treat empty string / "nan" as missing
    if s == "" or s.lower() == "nan":
        return None

    # Keep digits only (removes hyphens, spaces, etc.)
    digits = re.sub(r"\D", "", s)

    # If we have <= 11 digits, left-pad to recover leading zeros
    if 1 <= len(digits) <= 11:
        return digits.zfill(11)

    # Anything longer than 11 digits is not a valid NDC representation
    return None

# ---------------------------------------------------------
# Generate possible package NDC formats
# ---------------------------------------------------------
# openFDA `drug/label` stores NDCs in hyphenated form.
# Unfortunately, after you normalize to digits-only (11 digits),
# you cannot always know the *original* 10-digit display segmentation.
#
# To improve recall, we try several common hyphen segment-width patterns:
# - 5-4-2 (standard 11-digit packaging format)
# - 4-4-2 (some products are displayed this way)
# - 5-3-2 (another common 10-digit display pattern)
#
# If any variant hits, we accept it and stop early.
def ndc11_variants(ndc11):

    s = ndc11

    return [

        # 5-4-2
        f"{s[0:5]}-{s[5:9]}-{s[9:11]}",

        # 4-4-2
        f"{s[1:5]}-{s[5:9]}-{s[9:11]}",

        # 5-3-2
        f"{s[0:5]}-{s[6:9]}-{s[9:11]}"
    ]

# ---------------------------------------------------------
# FDA label archive lookup (openFDA)
# ---------------------------------------------------------
# Why `drug/label`?
# - Often includes records that are not present in the current NDC directory.
# - Can capture discontinued/historical products.
# - The `openfda` section frequently contains generic/brand/manufacturer metadata.
#
# What we do here:
# - Try several hyphenated NDC variants (see `ndc11_variants`).
# - Query openFDA with `limit=1` to keep responses small.
# - Return the *first* hit (we only need basic metadata, not all matches).
#
# Error handling:
# - We swallow exceptions per-variant to keep the pipeline moving.
# - Network/rate-limit issues are partially handled by the HTTPAdapter retries.
def fda_label_archive_lookup(ndc11):

    for v in ndc11_variants(ndc11):

        try:

            url = (
                f"https://api.fda.gov/drug/label.json"
                f"?search=openfda.package_ndc:\"{v}\"&limit=1"
            )

            r = http.get(url, timeout=10)
            r.raise_for_status()

            item = r.json().get("results", [None])[0]

            if item:

                return {
                    "source": "FDA_LABEL_ARCHIVE",
                    "variant": v,
                    "item": item
                }

        except Exception:
            # Keep going with the next variant.
            pass

    return None

# ---------------------------------------------------------
# RxNorm fallback lookup (RxNav)
# ---------------------------------------------------------
# Why RxNorm?
# - When openFDA does not return a label hit, RxNorm often still has a usable concept name.
# - This helps reduce the number of NOT_FOUND rows.
#
# Implementation notes:
# - `history=1` improves coverage for historical NDC mappings.
# - RxNav returns a structured object; we pick the best available "name".
# - We return a normalized hit object with the same shape as the FDA lookup.
def rxnorm_lookup(ndc11):

    try:

        url = (
            f"https://rxnav.nlm.nih.gov/REST/"
            f"ndcstatus.json?ndc={ndc11}&history=1"
        )

        r = http.get(url, timeout=10)
        r.raise_for_status()

        data = r.json().get("ndcStatus")

        if not data:
            return None

        # Choose the most informative name available.
        generic = (
            data.get("conceptName")
            or data.get("productName")
            or data.get("genericName")
        )

        if not generic:
            return None

        return {
            "source": "RXNORM",
            "item": {
                "generic_name": generic
            }
        }

    except Exception:
        # Treat any RxNav errors as "no hit" and let the pipeline continue.
        return None

# ---------------------------------------------------------
# Flatten API response into one output row
# ---------------------------------------------------------
# This function standardizes different upstream payload shapes into a single schema.
#
# Output schema:
# - NDC_11: the normalized 11-digit digits-only NDC
# - lookup_source: which lookup path produced the metadata
# - generic_name / brand_name / labeler_name: best-effort strings
#
# Notes:
# - openFDA label `openfda.*` fields are often lists; we take the first element.
# - RxNorm only provides a generic/concept name in this simplified workflow.
def extract_metadata(ndc11, hit):

    # Initialize a stable row shape so downstream concatenation is easy.
    row = {
        "NDC_11": ndc11,
        "lookup_source": None,
        "generic_name": None,
        "brand_name": None,
        "labeler_name": None
    }

    # No hit from either FDA or RxNorm.
    if not hit:

        row["lookup_source"] = "NOT_FOUND"
        return row

    # Record provenance (where the metadata came from).
    row["lookup_source"] = hit.get("source")

    item = hit.get("item", {})

    # -----------------------------
    # FDA label archive payload
    # -----------------------------
    if hit["source"] == "FDA_LABEL_ARCHIVE":

        ofda = item.get("openfda", {})

        generic = ofda.get("generic_name")
        brand = ofda.get("brand_name")
        manufacturer = ofda.get("manufacturer_name")

        row["generic_name"] = (
            generic[0]
            if isinstance(generic, list) and len(generic) > 0
            else generic
        )

        row["brand_name"] = (
            brand[0]
            if isinstance(brand, list) and len(brand) > 0
            else brand
        )

        row["labeler_name"] = (
            manufacturer[0]
            if isinstance(manufacturer, list) and len(manufacturer) > 0
            else manufacturer
        )

        return row

    # -----------------------------
    # RxNorm payload
    # -----------------------------
    if hit["source"] == "RXNORM":

        row["generic_name"] = item.get("generic_name")
        return row

    # Should not happen in this notebook, but keep a safe fallback.
    return row

# ---------------------------------------------------------
# Read input Excel
# ---------------------------------------------------------
# Read as strings so Excel does not coerce NDCs into numbers.
df_in = pd.read_excel(
    INPUT_XLSX,
    dtype=str
)

# Hard requirement for this notebook: the column name is known.
# If you need a different column name, change NDC_COLUMN above.
if NDC_COLUMN not in df_in.columns:

    raise ValueError(
        f"Expected column '{NDC_COLUMN}' "
        f"in {INPUT_XLSX}. Columns: {list(df_in.columns)}"
    )

# ---------------------------------------------------------
# Normalize NDC values
# ---------------------------------------------------------
# Convert each cell to an 11-digit digits-only string.
# - Strips non-digits
# - Left-pads to 11 digits to recover leading zeros
# - Returns None for invalid/unusable cells
ndc11_all = (
    df_in[NDC_COLUMN]
    .apply(clean_ndc11)
    .dropna()
    .unique()
    .tolist()
)

# ---------------------------------------------------------
# Split into chunks
# ---------------------------------------------------------
chunks = [

    ndc11_all[i:i + CHUNK_SIZE]

    for i in range(
        0,
        len(ndc11_all),
        CHUNK_SIZE
    )
]

# ---------------------------------------------------------
# Main chunk loop
# ---------------------------------------------------------
for chunk_idx, ndc_chunk in enumerate(chunks, 1):

    out_file = os.path.join(
        OUT_DIR,
        f"ndc_metadata_chunk_{chunk_idx:03d}.csv"
    )

    # Resume-safe behavior
    if os.path.exists(out_file):

        print(f"Chunk {chunk_idx} exists, skipped.")

        continue

    rows = []

    print(
        f"\nProcessing chunk "
        f"{chunk_idx}/{len(chunks)} "
        f"(size={len(ndc_chunk)})"
    )

    for ndc11 in ndc_chunk:

        # 1) openFDA label archive (fast, good coverage for many active + historical products)
        hit = fda_label_archive_lookup(ndc11)

        # 2) RxNorm fallback (fills remaining gaps when FDA label data is missing)
        if not hit:
            hit = rxnorm_lookup(ndc11)

        # Convert the raw API payload into a stable, flat schema.
        row = extract_metadata(ndc11, hit)
        rows.append(row)

        # Polite API pacing: reduces burst pressure on public endpoints.
        time.sleep(0.05)

    pd.DataFrame(rows).to_csv(
        out_file,
        index=False,
        encoding="utf-8-sig"
    )

print(
    "\nDone: "
    "NDC -> generic_name extraction complete."
)